# 2 · Entrenamiento y evaluación

> **Tipo de ML:** `supervisado`  · Modelo activo: `RandomForest`


---
> !  **Entorno**: ejecuta este notebook **dentro del entorno virtual** del proyecto.
> Si usas Jupyter directamente:
> ```bash
> make lab          # JupyterLab
> # o
> make notebook     # Jupyter Notebook clásico
> ```
> Ambos comandos lanzan Jupyter con `uv run`, garantizando que el paquete y sus dependencias
> están disponibles. **No abras el notebook desde fuera del entorno.**
---

## 0. Entorno


In [ ]:
# Verificar que el entorno está activo
import sys
print(f'Python: {sys.version}')


import xgboost; print(f'XGBoost: {xgboost.__version__}')



## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from climasafeai.utils.paths import (
    PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, REPORTS_DIR,
)


from climasafeai.models.train_model import train_models, load_models
from climasafeai.models.predict_model import evaluate_models, predict_new, DECISION_THRESHOLD
from climasafeai.visualization.visualize import plot_feature_importance


## 2. Cargar datos procesados


In [ ]:
# ── Carga de datos procesados ───────────────────────────────────────────────
# Si el pipeline aún no se ha ejecutado, corre primero: make data && make features
try:
    
    X_train = pd.read_csv(PROCESSED_DATA_DIR / 'X_train.csv')
    X_test  = pd.read_csv(PROCESSED_DATA_DIR / 'X_test.csv')
    y_train = pd.read_csv(PROCESSED_DATA_DIR / 'y_train.csv').squeeze()
    y_test  = pd.read_csv(PROCESSED_DATA_DIR / 'y_test.csv').squeeze()
    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
    print(f'Clases: {sorted(y_train.unique().tolist())}')
    print(f'Balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}')
    
except FileNotFoundError as _e:
    raise FileNotFoundError(
        f"Datos no encontrados: {_e}\n"
        "Ejecuta primero: make data && make features\n"
        "O desde este notebook: !make data features"
    ) from _e


## 3. Configuración


Modelo activo: `RandomForest`. Cambia `model_type` en json y regenera para entrenar otro.



In [ ]:

# Configuración: model_type activo = 'RandomForest'
# tune_knn=True  → busca el k óptimo para KNN (lento en datasets grandes)
# cv_evaluate=True → muestra F1 5-fold de cada modelo antes de guardar
TUNE_KNN    = True
CV_EVALUATE = True


## 4. Entrenar


In [ ]:

models = train_models(X_train, y_train, tune_knn=TUNE_KNN, cv_evaluate=CV_EVALUATE)


## 5. Evaluar


In [ ]:

threshold  = DECISION_THRESHOLD   # ajustar si clases muy desbalanceadas
df_results = evaluate_models(
    models, X_train, y_train, X_test, y_test, threshold=threshold
)
df_results.sort_values('F1_test', ascending=False)


## 6. Importancia de variables


In [ ]:

feature_names = X_train.columns.tolist()
plot_feature_importance(models, feature_names)
from IPython.display import Image
Image(FIGURES_DIR / 'feature_importance.png')


## 7. Predicción sobre nuevos datos


In [ ]:

best_name  = df_results.sort_values('F1_test', ascending=False).iloc[0]['Modelo']
best_model = models[best_name]
print(f'Mejor modelo: {best_name}')

# Predecir sobre nuevos datos:
# from climasafeai.features.build_features import process_input
# X_new   = process_input(df_nuevos)
# preds, probs = predict_new(best_model, X_new, threshold=threshold)


## 8. Profiling (opcional)

Si el entrenamiento es lento, identifica dónde se va el tiempo:

```bash
make profile
snakeviz reports/profile.prof
```
